## Parte 1 Preprocesamiento y Codificación
Objetivo: Limpiar definitivamente los datos detectados en el EDA y convertir todo a números para que el algoritmo pueda leerlos.

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ===========================================================
# FASE 3 - PREPROCESAMIENTO DE DATOS
# Objetivo:
# Preparar el dataset para técnicas de reducción de dimensionalidad,
# asegurando calidad, consistencia y homogeneidad en las variables.
# ===========================================================

# -----------------------------------------------------------
# 1. CARGA Y PREPARACIÓN INICIAL
# -----------------------------------------------------------

# Se carga el dataset original. El separador es tabulación (\t)
df = pd.read_csv('../data/marketing_campaign.csv', sep='\t')

# A partir del año de nacimiento se construye la variable "Age",
# la cual resulta más interpretable y útil para el análisis.
df['Age'] = 2024 - df['Year_Birth']

# Antes de aplicar cualquier filtro, se gestionan los valores faltantes.
# En este caso, la variable Income presenta algunos nulos (~1%),
# por lo que se imputan utilizando la mediana para evitar sesgos
# producidos por valores extremos.
df['Income'] = df['Income'].fillna(df['Income'].median())

# -----------------------------------------------------------
# 2. LIMPIEZA DE DATOS (SEGÚN HALLAZGOS DEL EDA)
# -----------------------------------------------------------

# Se eliminan registros con edades no realistas (mayores a 100 años),
# identificados previamente como errores de captura.
df = df[df['Age'] < 100]

# Se filtran ingresos extremadamente altos (outliers).
# Estos valores pueden distorsionar técnicas sensibles a la escala,
# como PCA, afectando la interpretación de los componentes.
df = df[df['Income'] < 200000]

# Se eliminan variables que no aportan valor al modelado:
# - ID: identificador único sin relevancia analítica
# - Dt_Customer: variable temporal sin transformación
# - Year_Birth: reemplazada por la variable Age
df_prep = df.drop(['ID', 'Dt_Customer', 'Year_Birth'], axis=1)

# -----------------------------------------------------------
# 3. CODIFICACIÓN DE VARIABLES CATEGÓRICAS
# -----------------------------------------------------------

# La variable Education se trata como ordinal,
# por lo que se utiliza Label Encoding para conservar su jerarquía.
le = LabelEncoder()
df_prep['Education'] = le.fit_transform(df_prep['Education'])

# La variable Marital_Status es de tipo nominal (sin orden inherente).
# Se aplica One-Hot Encoding para evitar introducir relaciones artificiales.
df_prep = pd.get_dummies(df_prep, columns=['Marital_Status'])

# -----------------------------------------------------------
# 4. ESCALADO DE VARIABLES
# -----------------------------------------------------------

# Se aplica estandarización (media = 0, desviación estándar = 1).
# Este paso es crítico para PCA, ya que evita que variables con
# mayor magnitud dominen el análisis.
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_prep)

# Se convierte nuevamente a DataFrame para facilitar su interpretación
# y asegurar compatibilidad con la siguiente fase (PCA).
df_scaled = pd.DataFrame(df_scaled, columns=df_prep.columns)

# -----------------------------------------------------------
# 5. VALIDACIÓN FINAL
# -----------------------------------------------------------

print("Preprocesamiento completado correctamente.")
print(f"Número de registros: {df_scaled.shape[0]}")
print(f"Número de variables: {df_scaled.shape[1]}")

Preprocesamiento completado correctamente.
Número de registros: 2236
Número de variables: 34
